# ItemCF：算法、评估与推荐

先完成 `01_data_analysis.ipynb`，再在项目根目录运行 `python train.py --model itemcf`。
本 Notebook 读取训练结果，并支持修改用户 ID 查看推荐。

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
                    if (path / 'src' / 'models' / 'item_cf.py').is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.recommend import file_sha256, recommend_for_user

TABLE_DIR = PROJECT_ROOT / 'outputs' / 'tables'
run_path = TABLE_DIR / 'itemcf_run.json'
if not run_path.is_file():
    raise FileNotFoundError('请先运行 python train.py --model itemcf。')
run = json.loads(run_path.read_text(encoding='utf-8'))
for name, checksum in run['input_sha256'].items():
    if file_sha256(PROJECT_ROOT / 'data' / 'processed' / name) != checksum:
        raise ValueError('预处理输入已变化，请重新运行 python train.py --model itemcf。')

## 1. 算法与参数选择

ItemCF 根据用户共同喜欢的电影计算相似度。训练输入是评分 ≥4 的二值矩阵 $B$，$C=B^TB$，$f_i=C_{ii}$：

$$s_{ij}=\frac{C_{ij}}{\sqrt{f_i f_j}}\frac{C_{ij}}{C_{ij}+\lambda},\qquad score(u,j)=\sum_{i\in H_u^+}s_{ij}.$$

自身相似度设为 0；每部历史电影保留最相似的 `n_neighbors` 部电影，再累加用户高分历史的相似度。
$\lambda$ 是收缩参数，0 表示普通余弦。推荐分数用于排序，不是预测星级；推荐时过滤所有已评分电影。
下表按验证集 **NDCG@10** 选择参数，测试集不参与选择。

In [2]:
search = pd.read_csv(TABLE_DIR / 'itemcf_validation_search.csv')
display(search[['n_neighbors', 'shrinkage', 'NDCG@10', 'selected']]
        .style.format({'shrinkage': '{:g}', 'NDCG@10': '{:.4f}'}))

,n_neighbors,shrinkage,NDCG@10,selected
0,20,0,0.0842,False
1,20,10,0.0835,False
2,20,50,0.0805,False
3,50,0,0.0844,True
4,50,10,0.0827,False
5,50,50,0.0790,False
6,100,0,0.0831,False
7,100,10,0.0812,False
8,100,50,0.0789,False


## 2. 评估结果

在完整未评分候选目录上评估前 10 部推荐，各指标先按用户计算再取平均：

- **Precision@10**：命中数 ÷ 10；**MeanHits@10** 为平均命中部数。
- **Recall@10**：命中数 ÷ 该用户留出集中的高分目标数。
- **NDCG@10**：考虑命中位置，越靠前越好，并按理想排序归一化。

验证使用训练历史，测试使用训练加验证历史，模型参数仅在训练集拟合。
未评分电影不计为已知命中，但其真实偏好未知；指标不等于实际满意度。

In [3]:
metrics = pd.read_csv(TABLE_DIR / 'itemcf_metrics.csv')
display(metrics[['split', 'users', 'Precision@10', 'Recall@10', 'NDCG@10', 'MeanHits@10']]
        .style.format({'Precision@10': '{:.2%}', 'Recall@10': '{:.2%}',
                       'NDCG@10': '{:.4f}', 'MeanHits@10': '{:.3f}'}))

,split,users,Precision@10,Recall@10,NDCG@10,MeanHits@10
0,validation,5776,5.97%,7.89%,0.0844,0.597
1,test,5765,5.64%,7.12%,0.0773,0.564


## 3. 按用户推荐电影

修改 `USER_ID` 后运行下面的单元格，显示用户尚未评分电影的标题和类型。
默认推荐 5 部，满足项目要求；演示使用测试期开始前的已知历史。

In [4]:
USER_ID = 1
TOP_K = 5

recommendations = recommend_for_user(USER_ID, stage='test', k=TOP_K)
display(recommendations[['rank', 'title', 'genres']])

,rank,title,genres
0,1,Raiders of the Lost Ark (1981),Action|Adventure
1,2,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Drama|Sci-Fi|War
2,3,"Shawshank Redemption, The (1994)",Drama
3,4,"Silence of the Lambs, The (1991)",Drama|Thriller
4,5,Pulp Fiction (1994),Crime|Drama


算法参考：Sarwar et al. (2001), [Item-based Collaborative Filtering Recommendation Algorithms](https://ra.ethz.ch/cdstore/www10/papers/519/sdm2.html)。本项目采用正反馈二值输入的 Top-N 变体。